<a href="https://colab.research.google.com/github/Kazi-Refat/PyTorch-Waste-Classifier/blob/main/Waste_Classification_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# =========================
# Waste Classification (PyTorch)
# Binary: Organic vs Recyclable
# =========================
bold text

In [ ]:
import os, time, copy, random
import numpy as np
from pathlib import Path
import contextlib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models


In [ ]:
# -------------------------
# Reproducibility (optional)
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # keep True only if you need absolute reproducibility
    torch.backends.cudnn.benchmark = True

set_seed(42)


In [ ]:
# -------------------------
# Config
# -------------------------
DATA_DIR = "/content/drive/MyDrive/DATASET"   # <-- change if needed
BATCH_SIZE = 32
NUM_EPOCHS = 10
IMG_SIZE = 224                     # 224 for ResNet; (use 299 if you switch to InceptionV3)
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
SAVE_PATH = "waste_resnet18_best.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [ ]:
# -------------------------
# Transforms (ImageNet stats)
# -------------------------
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -------------------------
# Datasets & Dataloaders
# -------------------------
def make_loader(split, tfm, shuffle, batch_size=BATCH_SIZE):
    dset = datasets.ImageFolder(root=os.path.join(DATA_DIR, split), transform=tfm)
    loader = DataLoader(dset, batch_size=batch_size, shuffle=shuffle,
                        num_workers=NUM_WORKERS, pin_memory=True)
    return dset, loader

train_set, train_loader = make_loader("/content/drive/MyDrive/DATASET/TRAIN", train_tfms, shuffle=True)
val_set,   val_loader   = make_loader("/content/drive/MyDrive/DATASET/TEST",   eval_tfms,   shuffle=False)

# (Optional) test loader, if you have a test split
test_set,  test_loader  = None, None
test_path = Path('/content/drive/MyDrive/DATASET/TEST') / "test"
if test_path.exists():
    test_set, test_loader = make_loader("test", eval_tfms, shuffle=False)

class_names = train_set.classes
num_classes = len(class_names)
print("Classes:", class_names, "| Train images:", len(train_set), "| Val images:", len(val_set))

Classes: ['O', 'R'] | Train images: 22594 | Val images: 2513


In [ ]:
# -------------------------
# Model 1: ResNet18 (pretrained)
# -------------------------
model1 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # Load pre-trained ResNet18

# Replace the final fully connected layer
num_ftrs = model1.fc.in_features
model1.fc = nn.Linear(num_ftrs, num_classes)

# Move the model to the appropriate device (GPU or CPU)
model1 = model1.to(DEVICE)

# -------------------------
# Loss, Optimizer, Scheduler
# -------------------------
# Handle potential class imbalance by using class weights (optional)
def get_class_weights(dataset):
    targets = np.array(dataset.targets)
    counts = np.bincount(targets, minlength=num_classes)
    weights = counts.sum() / (counts + 1e-9)
    weights = weights / weights.mean()  # normalize
    return torch.tensor(weights, dtype=torch.float32)

use_class_weights = True
if use_class_weights:
    class_weights = get_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]
/tmp/ipython-input-180153571.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


In [ ]:
# -------------------------
# Train / Validate
# -------------------------
def run_epoch(model, loader, phase="train"):
    is_train = phase == "train"
    model.train(is_train)

    running_loss = 0.0
    running_corrects = 0
    n_samples = 0

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels).item()
        n_samples += inputs.size(0)

    epoch_loss = running_loss / n_samples
    epoch_acc = running_corrects / n_samples
    return epoch_loss, epoch_acc

best_acc = 0.0
best_wts = copy.deepcopy(model1.state_dict())
early_stop_patience = 5
no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(model1, train_loader, phase="train")
    val_loss,   val_acc   = run_epoch(model1, val_loader,   phase="val")
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model1.state_dict())
        torch.save(best_wts, SAVE_PATH)
        no_improve = 0
    else:
        no_improve += 1

    dur = time.time() - start
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f} | "
          f"Best Val Acc {best_acc:.4f} | "
          f"{dur:.1f}s")

    if no_improve >= early_stop_patience:
        print("Early stopping triggered.")
        break

# Load best weights
model1.load_state_dict(best_wts)

Epoch 01/10 | Train Loss 0.3280 Acc 0.8702 | Val Loss 0.2338 Acc 0.9093 | Best Val Acc 0.9093 | 2788.2s
Epoch 02/10 | Train Loss 0.2898 Acc 0.8873 | Val Loss 0.2462 Acc 0.9033 | Best Val Acc 0.9093 | 221.5s
Epoch 03/10 | Train Loss 0.2403 Acc 0.9074 | Val Loss 0.3193 Acc 0.8615 | Best Val Acc 0.9093 | 225.7s
Epoch 04/10 | Train Loss 0.2206 Acc 0.9154 | Val Loss 0.2160 Acc 0.9268 | Best Val Acc 0.9268 | 223.8s
Epoch 05/10 | Train Loss 0.2115 Acc 0.9183 | Val Loss 0.2221 Acc 0.9208 | Best Val Acc 0.9268 | 227.2s
Epoch 06/10 | Train Loss 0.1767 Acc 0.9319 | Val Loss 0.1875 Acc 0.9304 | Best Val Acc 0.9304 | 224.6s
Epoch 07/10 | Train Loss 0.1594 Acc 0.9404 | Val Loss 0.1875 Acc 0.9355 | Best Val Acc 0.9355 | 226.4s
Epoch 08/10 | Train Loss 0.1325 Acc 0.9499 | Val Loss 0.1960 Acc 0.9363 | Best Val Acc 0.9363 | 228.6s
Epoch 09/10 | Train Loss 0.1120 Acc 0.9579 | Val Loss 0.1713 Acc 0.9379 | Best Val Acc 0.9379 | 227.4s
Epoch 10/10 | Train Loss 0.0949 Acc 0.9628 | Val Loss 0.2448 Acc 0.9308 

<All keys matched successfully>

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, mean_squared_error
import numpy as np # Import numpy for sqrt

# Updated evaluation function to compute multiple metrics
def evaluate(loader, model, criterion, phase="test"):
    model.eval()  # Set model to evaluation mode
    all_preds = []
    all_labels = []
    running_loss = 0.0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                outputs = model(inputs)  # Get model outputs (InceptionV3 returns one output during evaluation)
                loss = criterion(outputs, labels)

            # Convert logits to predictions
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())  # Store predictions
            all_labels.extend(labels.cpu().numpy())  # Store true labels
            running_loss += loss.item() * inputs.size(0)

    # Calculate evaluation metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_precision = precision_score(all_labels, all_preds, average='binary')  # Precision for binary classification
    epoch_recall = recall_score(all_labels, all_preds, average='binary')  # Recall for binary classification
    mse = mean_squared_error(all_labels, all_preds) # Calculate Mean Squared Error
    epoch_rmse = np.sqrt(mse)  # Calculate RMSE by taking the square root of MSE

    # Print the evaluation results
    print(f"{phase} Loss: {epoch_loss:.4f} | "
          f"Acc: {epoch_acc:.4f} | "
          f"Precision: {epoch_precision:.4f} | "
          f"Recall: {epoch_recall:.4f} | "
          f"RMSE: {epoch_rmse:.4f}")

    return epoch_acc, epoch_precision, epoch_recall, epoch_rmse, epoch_loss

In [ ]:
# -------------------------
# Evaluate on Test Set
# -------------------------
if test_loader is not None:
    test_acc, test_precision, test_recall, test_rmse, test_loss = evaluate(test_loader, model1, criterion, phase="Test")
    print(f"Test Results: Acc: {test_acc:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, RMSE: {test_rmse:.4f}")

# -------------------------
# Evaluate on Validation Set
# -------------------------
val_acc, val_precision, val_recall, val_rmse, val_loss = evaluate(val_loader, model1, criterion, phase="Validation")
print(f"Validation Results: Acc: {val_acc:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, RMSE: {val_rmse:.4f}")

Validation Loss: 0.1713 | Acc: 0.9379 | Precision: 0.9509 | Recall: 0.9065 | RMSE: 0.2492
Validation Results: Acc: 0.9379, Precision: 0.9509, Recall: 0.9065, RMSE: 0.2492


In [ ]:
# ======================
# Save model checkpoint
# ======================
save_model = True
root_path = "./data"   # or any folder you like

if save_model:
    torch.save(model1.state_dict(), root_path + "waste_classifier.pth")
    print("✅ Model saved at", root_path + "waste_classifier.pth")

✅ Model saved at ./datawaste_classifier.pth


In [ ]:
# ======================
# Load trained model
# ======================
load_model = True
root_path = "./data"

if load_model:
    # rebuild the same architecture first
    from torchvision import models
    import torch.nn as nn

    model1 = models.resnet18(weights=None)
    model1.fc = nn.Linear(model1.fc.in_features, num_classes)

    model1.load_state_dict(torch.load(root_path + "waste_classifier.pth", map_location=DEVICE))
    model1.to(DEVICE).eval()
    print("✅ Trained model loaded")


✅ Trained model loaded


In [ ]:

# -------------------------
# Predict single image helper
# -------------------------
from PIL import Image

def predict_image(img_path, topk=2):
    model1.eval()
    img = Image.open(img_path).convert("RGB")
    x = eval_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        logits = model1(x)
        probs = torch.softmax(logits, dim=1)[0]
    topk_probs, topk_idx = probs.topk(topk)
    for p, idx in zip(topk_probs.tolist(), topk_idx.tolist()):
        print(f"{class_names[idx]}: {p*100:.2f}%")

# Example:
predict_image("/content/drive/MyDrive/DATASET/palstic.jpg")


R: 95.92%
O: 4.08%


/tmp/ipython-input-2791274664.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


Model: InceptionV3 (pretrained)

In [ ]:
# -------------------------
# Model 2: InceptionV3 (pretrained)
# -------------------------
# --- Model ---
# Use the modern weights API and keep aux logits enabled for training-time aux loss
model2 = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, aux_logits=True)
num_features_main = model2.fc.in_features
num_features_aux  = model2.AuxLogits.fc.in_features

model2.fc = nn.Linear(num_features_main, num_classes)
model2.AuxLogits.fc = nn.Linear(num_features_aux,  num_classes)
model2 = model2.to(DEVICE)

IMG_SIZE = 299  # InceptionV3 expects 299x299

# --- Transforms ---
# IMPORTANT: Do NOT Resize(299) before RandomResizedCrop(299).
# Ensure RGB (handles grayscale inputs).
to_rgb = transforms.Lambda(lambda x: x.convert("RGB"))

train_tfms = transforms.Compose([
    to_rgb,
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(3/4, 4/3)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# Canonical eval pipeline: resize a bit larger, then center-crop to 299
eval_tfms = transforms.Compose([
    to_rgb,
    transforms.Resize(int(IMG_SIZE * 342 / 299)),   # ~342
    transforms.CenterCrop(IMG_SIZE),                # 299
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# --- Loss / Optim / Scheduler ---
criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = optim.AdamW(model2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Mixed Precision training (optional)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 211MB/s]
/tmp/ipython-input-667140794.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


In [ ]:
# --- Train/Val epoch ---
import torch.nn.functional as F
"""
_ensure_299: some batches aren't reaching InceptionV3 as 3×299×299 RGB.
Because a few images come in smaller (or occasionally 1-channel), the spatial map shrinks through the early downsamplings to 3×3, and a later 5×5 convolution crashes (kernel 5×5 > input 3×3).

How to avoid it :
Guarantee shape: force inputs to [B,3,299,299] right before the forward pass.
Safe transforms: use RandomResizedCrop(299) for train; Resize(342)→CenterCrop(299) for val/test; and convert to RGB.
Optional sanity check: print a batch once (print(x.shape)) and you'll likely see a mismatch on the failing run.
"""
def _ensure_299(inputs: torch.Tensor) -> torch.Tensor:
    """
    Make sure input is [B,3,299,299]. If not, upsample to (299,299).
    This prevents deep layers from collapsing to too-small feature maps.
    """
    # Enforce 3 channels just in case some batch contains grayscale (C=1)
    if inputs.shape[1] == 1:
        inputs = inputs.repeat(1, 3, 1, 1)
    # Enforce spatial size 299x299
    if inputs.shape[-2] != 299 or inputs.shape[-1] != 299:
        inputs = F.interpolate(inputs, size=(299, 299), mode="bilinear", align_corners=False)
    return inputs

# Unified autocast context across PyTorch versions
def _autocast(device_type: str, enabled: bool = True):
    """
    Use torch.amp.autocast(device_type=...) when available (PyTorch >= 2.0).
    Fall back to torch.cuda.amp.autocast on older versions for CUDA, or no-op on CPU.
    """
    # New API present?
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type=device_type, enabled=enabled)

    # Fallbacks for older versions (no device_type kwarg)
    if device_type == "cuda":
        return torch.cuda.amp.autocast(enabled=enabled)
    else:
        return contextlib.nullcontext()


# ---- One epoch (train/val) ----
def run_epoch(model, loader, phase="train", aux_weight: float = 0.4):
    """
    Assumes the following exist in scope:
      - optimizer, criterion, scaler (for mixed precision), DEVICE
    """
    is_train = (phase == "train")
    model.train(is_train)

    running_loss = 0.0
    running_corrects = 0
    n_samples = 0

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # FINAL GUARANTEE: make inputs safe for InceptionV3
        inputs = _ensure_299(inputs)

        optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            device_type = "cuda" if DEVICE.type == "cuda" else "cpu"
            with _autocast(device_type=device_type, enabled=True):
                outputs = model(inputs)

                # During training, InceptionV3 returns (logits, aux_logits).
                # During eval, it returns just logits.
                if is_train and isinstance(outputs, (tuple, list)):
                    logits, aux_logits = outputs
                    loss = criterion(logits, labels) + aux_weight * criterion(aux_logits, labels)
                else:
                    logits = outputs
                    loss = criterion(logits, labels)

            _, preds = torch.max(logits, 1)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss     += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels).item()
        n_samples        += inputs.size(0)

    epoch_loss = running_loss / max(n_samples, 1)
    epoch_acc  = running_corrects / max(n_samples, 1)
    return epoch_loss, epoch_acc


# ---- Training loop with early stopping (assumes: model2, train_loader, val_loader,
#      optimizer, criterion, scaler, scheduler, NUM_EPOCHS, SAVE_PATH already defined) ----
best_acc = 0.0
best_wts = copy.deepcopy(model2.state_dict())
early_stop_patience = 5
no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(model2, train_loader, phase="train")
    val_loss,   val_acc   = run_epoch(model2, val_loader,   phase="val")
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model2.state_dict())
        torch.save(best_wts, SAVE_PATH)
        no_improve = 0
    else:
        no_improve += 1

    dur = time.time() - start
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f} | "
          f"Best Val Acc {best_acc:.4f} | "
          f"{dur:.1f}s")

    if no_improve >= early_stop_patience:
        print("Early stopping triggered.")
        break

# Load best weights
model2.load_state_dict(best_wts)

Epoch 01/10 | Train Loss 0.4596 Acc 0.8701 | Val Loss 0.2419 Acc 0.8969 | Best Val Acc 0.8969 | 298.5s
Epoch 02/10 | Train Loss 0.3789 Acc 0.8962 | Val Loss 0.2142 Acc 0.9264 | Best Val Acc 0.9264 | 270.2s
Epoch 03/10 | Train Loss 0.3313 Acc 0.9076 | Val Loss 0.2637 Acc 0.9268 | Best Val Acc 0.9268 | 269.8s
Epoch 04/10 | Train Loss 0.3087 Acc 0.9156 | Val Loss 0.2001 Acc 0.9244 | Best Val Acc 0.9268 | 271.7s
Epoch 05/10 | Train Loss 0.2693 Acc 0.9264 | Val Loss 0.2084 Acc 0.9216 | Best Val Acc 0.9268 | 270.5s
Epoch 06/10 | Train Loss 0.2473 Acc 0.9337 | Val Loss 0.2172 Acc 0.9196 | Best Val Acc 0.9268 | 269.2s
Epoch 07/10 | Train Loss 0.2144 Acc 0.9428 | Val Loss 0.2105 Acc 0.9280 | Best Val Acc 0.9280 | 273.9s
Epoch 08/10 | Train Loss 0.1804 Acc 0.9526 | Val Loss 0.2220 Acc 0.9292 | Best Val Acc 0.9292 | 272.5s
Epoch 09/10 | Train Loss 0.1469 Acc 0.9609 | Val Loss 0.1775 Acc 0.9419 | Best Val Acc 0.9419 | 273.0s
Epoch 10/10 | Train Loss 0.1213 Acc 0.9671 | Val Loss 0.2121 Acc 0.9327 |

<All keys matched successfully>

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_squared_error
import numpy as np # Import numpy for sqrt

def evaluate(loader, model, criterion, phase="test"):
    model.eval()  # Set model to evaluation mode
    all_preds = []
    all_labels = []
    running_loss = 0.0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                outputs = model(inputs)  # Get model outputs
                loss = criterion(outputs, labels)

            # Convert outputs to probabilities (for precision, recall, etc.)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())  # Store predictions
            all_labels.extend(labels.cpu().numpy())  # Store true labels
            running_loss += loss.item() * inputs.size(0)

    # Calculate the evaluation metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_precision = precision_score(all_labels, all_preds, average='binary')  # assuming binary classification
    epoch_recall = recall_score(all_labels, all_preds, average='binary')  # assuming binary classification
    mse = mean_squared_error(all_labels, all_preds) # Calculate Mean Squared Error
    epoch_rmse = np.sqrt(mse)  # Calculate RMSE by taking the square root of MSE

    # Print results
    print(f"{phase} Loss: {epoch_loss:.4f} | "
          f"Acc: {epoch_acc:.4f} | "
          f"Precision: {epoch_precision:.4f} | "
          f"Recall: {epoch_recall:.4f} | "
          f"RMSE: {epoch_rmse:.4f}")

    return epoch_acc, epoch_precision, epoch_recall, epoch_rmse, epoch_loss

# Usage example:
# test_acc, test_precision, test_recall, test_rmse, test_loss = evaluate(test_loader, model2, criterion, "test")

In [ ]:
# Evaluate on Test Set
if test_loader is not None:
    test_acc, test_precision, test_recall, test_rmse, test_loss = evaluate(test_loader, model2, criterion, phase="Test")
    print(f"Test Results: Acc: {test_acc:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, RMSE: {test_rmse:.4f}")

# Evaluate on Validation Set
val_acc, val_precision, val_recall, val_rmse, val_loss = evaluate(val_loader, model2, criterion, phase="Validation")
print(f"Validation Results: Acc: {val_acc:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, RMSE: {val_rmse:.4f}")

Validation Loss: 0.2400 | Acc: 0.9176 | Precision: 0.9080 | Recall: 0.9056 | RMSE: 0.2870
Validation Results: Acc: 0.9176, Precision: 0.9080, Recall: 0.9056, RMSE: 0.2870


In [ ]:

# -------------------------
# Predict single image helper
# -------------------------
from PIL import Image

def predict_image(img_path, topk=2):
    model2.eval()
    img = Image.open(img_path).convert("RGB")
    x = eval_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        logits = model2(x)
        probs = torch.softmax(logits, dim=1)[0]
    topk_probs, topk_idx = probs.topk(topk)
    for p, idx in zip(topk_probs.tolist(), topk_idx.tolist()):
        print(f"{class_names[idx]}: {p*100:.2f}%")

# Example:
predict_image("/content/drive/MyDrive/DATASET/palstic.jpg")


R: 99.75%
O: 0.25%


/tmp/ipython-input-2375154625.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
